# Notebook 2 — Bronze Ingestion (XML)

**Run after profile_raw_xml.ipynb.** Use the same `xml_path` and `row_tag` from that notebook.

In [0]:
from pyspark.sql import functions as F

dbutils.widgets.text(
    "xml_path",
    "/Volumes/dev_automotive/landing/landing_geography/final_geografic.xml",
    "XML file path (volume)",
)
dbutils.widgets.text("row_tag", "record", "rowTag (from Notebook 1)")
dbutils.widgets.text(
    "bronze_table",
    "dev_automotive.bronze.geografic",
    "Bronze table (catalog.schema.table)",
)
dbutils.widgets.dropdown("run_healer", "false", ["true", "false"], "Run healer (only when XML has malformed empty tags <>value</>)")

xml_path = dbutils.widgets.get("xml_path").strip()
row_tag = dbutils.widgets.get("row_tag").strip()
bronze_table = dbutils.widgets.get("bronze_table").strip()
run_healer = dbutils.widgets.get("run_healer").strip().lower() == "true"


## 1. Healer (only when malformed)

Run only when XML has malformed empty tags (e.g. `<>0</>`). Otherwise skip; Reader will use `xml_path` directly. 


In [0]:
# Only when run_healer: fix <>value</> → <index>value</index>, write to temp path
if run_healer:
    temp_path = f"{xml_path}_healed"
    (
        spark.read.text(xml_path)
        .withColumn("value", F.regexp_replace("value", r"<>([^<]*)</>", r"<index>$1</index>"))
        .write.mode("overwrite")
        .text(temp_path)
    )
    read_path = temp_path
    print(f"✅ Structural fix applied and saved to: {read_path}")
else:
    read_path = xml_path
    print("Healer skipped; reading from source path.")


## 2. Reader — load healed XML and infer schema

In [0]:
df_bronze = (
    spark.read
    .format("xml")
    .option("rowTag", row_tag)
    .option("inferSchema", "true")
    .load(read_path)
)


## 3. Row & column counts (profiling)

In [0]:
row_count = df_bronze.count()
col_count = len(df_bronze.columns)
print(f"📊 INGESTION SUMMARY: {row_count:,} rows | {col_count} columns detected")


In [0]:
(
    df_bronze
    .withColumn("bronze_ingested_at", F.current_timestamp())
    .withColumn("source_file", F.lit(xml_path))
    .write.format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(bronze_table)
)
print(f"✅ Written to {bronze_table}")

In [0]:
if run_healer:
    dbutils.fs.rm(temp_path, recurse=True)
    print("Temp path removed.")

In [0]:
display(spark.table(bronze_table))
